In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
from shapely import Polygon
from simple_modflow.modflow.mf6.voronoiplus import VoronoiGridPlus as Vor
from pathlib import Path
import bisect
import flopy
from simple_modflow.modflow.utils.datatypes.readers import read_shp_gpkg
import scipy
import pickle
import pandas as pd
import geopandas as gpd
import figs as f
from simple_modflow.modflow.mf6.mfsimbase import SimulationBase
from simple_modflow.modflow.utils.surfaces import InterpolatedSurface
from pandas import IndexSlice as idxx


In [2]:
from figs._aq_test import AquiferTestFigure
tst = pd.read_excel(Path(r"C:\Users\lukem\Python\data\xlsx\P465 step test.xlsx"))
fig = AquiferTestFigure()
fig.add_scattergl(x=tst.iloc[:, 0], y=tst.iloc[:, 1])
fig.write_scaled_pdf()

writing SVG to temp_plot.svg


C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\plotly\io\_kaleido.py:528: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




writing SVG to temp_plot.svg


C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\plotly\io\_kaleido.py:528: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




using cairo


MemoryError: ("cairo returned CAIRO_STATUS_NO_MEMORY: b'out of memory'", 1)

,0.000000,0.005038
0,0.1,0.009688
1,0.2,-0.008506
2,0.3,0.021349
3,0.4,0.022404
4,0.5,0.005454
...,...,...
3095,309.6,6.269205
3096,309.7,6.230410
3097,309.8,6.172351
3098,309.9,6.156375


In [2]:
with open(Path(r"C:\Users\lukem\mf6\LkPt_F5_flow_lk3\LkPt_F5_flow_lk3.model"), 'rb') as file:
    model: SimulationBase = pickle.load(file)

In [3]:
import flopy
from pathlib import Path

ws = model.model_output_folder_path
nam = model.name

sim = flopy.mf6.MFSimulation.load(
    sim_ws=str(ws),
    strict=False,             # be tolerant with old files
    verify_data=False,        # don't force-check every file
)

gwf = sim.get_model()        # or sim.get_model("your_model_name")

# now pass these into your wrapper instead of the old ones
mod = SimulationBase(name=nam)
mod.gwf = gwf
mod.vor = model.vor
vor = mod.vor
vor2 = Vor(verts=vor.verts, iverts=vor.iverts, xcyc=[vor.centroids_x, vor.centroids_y])
vor2.gdf_topbtm = vor.gdf_topbtm
mod.vor = vor2


loading simulation...
  loading simulation name file...
  loading tdis package...
  loading model gwf6...
    loading package disv...
    loading package npf...
    loading package oc...
    loading package ic...
    loading package sto...
    loading package rch...
    loading package drn...
    loading package ghb...
    loading package lak...
    loading package sfr...
  loading solution package lkpt_f5_flow_lk3...
VoronoiGrid initializing.
Voronoi grid initialized.


In [4]:
mod.cor(per=15, zmin=380, zmax=390).plot()

In [105]:
mod.lak.stage.get()

array([[374.34673557, 384.00028902],
       [374.33040587, 384.208884  ],
       [374.31444497, 384.07782804],
       [374.29881951, 384.25135818],
       [374.28359686, 384.22584851],
       [374.26873761, 384.05272262],
       [374.25416931, 384.25389553],
       [374.24000171, 384.36041191],
       [374.22629153, 384.03270055],
       [374.21281637, 384.01250719],
       [374.19955616, 384.2856555 ],
       [374.18672396, 384.48753217],
       [374.17457017, 384.37476968],
       [374.16286798, 384.0528322 ],
       [374.151312  , 384.11979997],
       [374.14141299, 389.34691481],
       [374.13495088, 386.42847358],
       [374.12898773, 385.48196265],
       [374.12310435, 384.55245747],
       [374.1170268 , 384.18325322],
       [374.1105725 , 384.10188234],
       [374.10376344, 384.07233956],
       [374.09663781, 384.05081626],
       [374.0892467 , 384.03593323],
       [374.08165436, 384.26263239],
       [374.07402772, 384.09355737],
       [374.06628506, 384.03800192],
 

In [59]:
vor = mod.vor
vor2 = Vor(verts=vor.verts, iverts=vor.iverts, xcyc=[vor.centroids_x, vor.centroids_y])
vor2.gdf_topbtm = vor.gdf_topbtm
mod.vor = vor2

VoronoiGrid initializing.
Voronoi grid initialized.


In [69]:
model.cor(per=16, zmin=380, zmax=387).plot()

In [5]:
gd: gpd.GeoDataFrame = pd.concat([mod.vor.gdf_vorPolys, pd.DataFrame(mod.gwf.npf.k.get_data()).T], axis=1)
gd.columns = ['geometry', '0', '1', '2']
gd.to_file('lakepoint_ks_fac5.gpkg', driver='GPKG')


In [10]:
mod.srf.lyr(layer=3).save_raster(output_tif=Path(r"C:\Users\lukem\mf6\LakePointe misc\Facility 5") / 'lakepoint_fac5_layer3_botm.tif')

In [ ]:
aq_tst = Path(r"C:\Users\lukem\Python\data\EB-118W test.xlsx")
pump_well = pd.read_excel(aq_tst).loc[:, ['min', 's']]
hand_data = pd.read_excel(aq_tst, sheet_name=1).loc[:, ['step', 'min', 'Q', 's feet']]
hand_data['step'] = hand_data['step'].ffill()
step_times = [hand_data[hand_data['step'] == stp].iloc[0].loc['min'] for stp in [1,2,3,4,5]]
step_times
def stp_times(x):
    step_idx = bisect.bisect(step_times, x) - 1
    return x - step_times[step_idx], step_idx
pump_well['step times'] = pump_well.loc[:, 'min'].apply(lambda x: stp_times(x)[0])
pump_well['step'] = pump_well.loc[:, 'min'].apply(lambda x: stp_times(x)[1]+1)

fig = f.Fig()
for stp in [1,2,3,4,5]:
    fig.add_scattergl(
        x=pump_well[pump_well['step'] == stp].loc[:, 'step times'],
        y=pump_well[pump_well['step'] == stp].loc[:, 's'],
        name=stp
    )
fig.update_layout(
    xaxis_type="log",
    #yaxis_type="log",
)
fig.show()
fig2 = f.Fig()
fig2.add_scattergl(
    y=hand_data['Q'].ffill().bfill(),
    x=hand_data['min'],
    name='Q'
)
fig2.add_scattergl(
    x=pump_well['min'],
    y=pump_well['s'],
)
fig2.show()

In [ ]:
from shiny.express import input, ui
from shinywidgets import render_plotly

ui.input_selectize(
    "var", "Select variable",
    choices=["bill_length_mm", "body_mass_g"]
)

@render_plotly
def hist():
    import plotly.express as px
    from palmerpenguins import load_penguins
    df = load_penguins()
    return px.histogram(df, x=input.var())

# Create the Shiny Express app
app = App(ui, None)


In [ ]:
from shiny.express import input, ui
from shinywidgets import render_plotly

ui.input_selectize(
    "var", "Select variable",
    choices=["bill_length_mm", "body_mass_g"]
)

@render_plotly
def hist():
    import plotly.express as px
    from palmerpenguins import load_penguins
    df = load_penguins()
    return px.histogram(df, x=input.var())

# Create the Shiny Express app
app = App(ui, None)
